# Sumativa 3 — Taller II

## Integración de fuentes heterogéneas con Apache Spark

**Curso:** Gestión de Datos y Tecnologías Big Data (MCDI502)  
**Grupo:** 7  
**Archivo:** `mcdi502_s3_grupo7.ipynb`

Este notebook sigue la rúbrica de la Sumativa 3 y adapta los ejemplos entregados por el docente. Cada sección indica qué fuente se conecta, qué variables incorpora y qué resultado debe mostrar.

## Ruta de trabajo

1. Preparar el entorno y crear SparkSession/SparkContext.
2. Obtener y comprobar los 100 registros requeridos de cada fuente.
3. Crear las columnas `altura_metros` y `condicion_clima`.
4. Integrar las cuatro fuentes mediante un índice de fila documentado.
5. Crear las cuatro vistas temporales y ejecutar C1–C7.

**Antes de ejecutar:** abre el notebook en Google Colab con una sesión nueva. Ten disponibles las credenciales de MongoDB del curso y, para consultar Santiago, un token AQICN con acceso a esa ubicación. El token `demo` permite reproducir el ejemplo docente, pero la estación devuelta debe revisarse: una respuesta de otra ciudad no cumple el requisito regional.

Si una fuente tiene menos de 100 filas, se muestra el número real. El requisito queda pendiente hasta disponer de los registros; no se completan con datos inventados.

## Fuentes y variables

| Fuente | DataFrame | Variables incluidas |
|---|---|---|
| OpenMeteo | `df_openmeteo` | Temperatura, humedad, precipitación, viento, dirección e índice UV |
| AQICN | `df_aqicn` | PM2.5, PM10, O3, NO2 y SO2 |
| MongoDB (ESP32) | `df_mongo_bpm` | Altura en pies, presión atmosférica, monóxido de carbono y fecha |
| MySQL (ESP32 vía API REST) | `df_mysql_dht` | Temperatura interior, humedad, identificador y fecha |

Los CSV de ventas incorporados como material adicional no se usan aquí porque no son parte de las cuatro fuentes exigidas por esta rúbrica.

## 1. Configuración del entorno

Se mantiene **PySpark 3.5.7** y los conectores del ejemplo docente de MongoDB. Spark se instala con `pip`, para evitar depender de una descarga de distribución que ya no está disponible en su URL original. Se utiliza **Java 17**, compatible con esta versión de Spark.

Ejecuta esta sección en una sesión nueva de Colab. La descarga de los dos JAR se detiene con un error si no puede completarse.

In [ ]:
!apt-get update -qq
!apt-get install -y -qq openjdk-17-jdk-headless > /dev/null
%pip install -q pyspark==3.5.7 findspark requests

from pathlib import Path
from urllib.request import urlopen

# Los mismos conectores empleados en el ejemplo docente.
conectores = {
    "mongo-spark-connector_2.12-3.0.1.jar":
        "https://repo1.maven.org/maven2/org/mongodb/spark/mongo-spark-connector_2.12/3.0.1/mongo-spark-connector_2.12-3.0.1.jar",
    "mongo-java-driver-3.12.10.jar":
        "https://repo1.maven.org/maven2/org/mongodb/mongo-java-driver/3.12.10/mongo-java-driver-3.12.10.jar"
}
for nombre, url in conectores.items():
    with urlopen(url, timeout=30) as respuesta:
        Path("/content", nombre).write_bytes(respuesta.read())

print("Conectores descargados. La siguiente sección comprueba Java y Spark.")

## 2. Importaciones y variables de entorno

Las columnas temporales se manejarán con la zona horaria de Santiago para que las consultas por fecha sean legibles.

In [ ]:
import os
import time
import requests
import pyspark
import findspark
from datetime import datetime, timedelta
from zoneinfo import ZoneInfo
from getpass import getpass

os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
os.environ["SPARK_HOME"] = str(Path(pyspark.__file__).parent)
if not Path(os.environ["JAVA_HOME"], "bin/java").is_file():
    raise RuntimeError("No se instaló Java 17. Revisa la salida de la celda anterior.")
findspark.init()

from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, when, round as spark_round, row_number, to_timestamp
)
from pyspark.sql.types import (
    StructType, StructField, StringType, DoubleType, TimestampType,
    IntegerType
)
from pyspark.sql.window import Window

ZONA_HORARIA = "America/Santiago"
CANTIDAD_OBJETIVO = 100
print("Java:", os.environ["JAVA_HOME"])
print("Spark:", os.environ["SPARK_HOME"])

In [ ]:
spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("MCDI502_Sumativa3_Grupo7")
    .config(
        "spark.jars",
        "/content/mongo-spark-connector_2.12-3.0.1.jar,"
        "/content/mongo-java-driver-3.12.10.jar"
    )
    .config("spark.sql.session.timeZone", ZONA_HORARIA)
    .getOrCreate()
)
sc = spark.sparkContext

print("Spark:", spark.version)
assert spark.range(1).count() == 1, "Spark no pudo ejecutar la comprobación inicial."
print("SparkSession y SparkContext operativos.")

## 3. Ingesta de datos — OpenMeteo

Se consultan datos horarios de Viña del Mar para los últimos siete días. La API entrega más de 100 horas; se seleccionan **100 observaciones repartidas a lo largo de la semana**. Así, C3 trabaja sobre registros de toda la semana y el DataFrame cumple el tamaño solicitado.

Las variables corresponden a las indicadas en la pauta. El nombre técnico de cada campo se conserva para mantener el código simple.

In [ ]:
LATITUD = -33.02457
LONGITUD = -71.55183
VARIABLES_OPENMETEO = [
    "temperature_2m",
    "relative_humidity_2m",
    "precipitation",
    "wind_speed_10m",
    "wind_direction_10m",
    "uv_index"
]

parametros_openmeteo = {
    "latitude": LATITUD,
    "longitude": LONGITUD,
    "hourly": ",".join(VARIABLES_OPENMETEO),
    "past_days": 7,
    "forecast_days": 1,
    "timezone": ZONA_HORARIA
}

respuesta = requests.get(
    "https://api.open-meteo.com/v1/forecast",
    params=parametros_openmeteo,
    timeout=30
)
respuesta.raise_for_status()
datos_openmeteo = respuesta.json()["hourly"]

ahora_santiago = datetime.now(ZoneInfo(ZONA_HORARIA))
inicio_semana = ahora_santiago - timedelta(days=7)

observaciones_openmeteo = []
for i, tiempo in enumerate(datos_openmeteo["time"]):
    fecha_hora = datetime.fromisoformat(tiempo).replace(tzinfo=ZoneInfo(ZONA_HORARIA))
    if inicio_semana <= fecha_hora <= ahora_santiago:
        fila = {"tiempo": fecha_hora}
        for variable in VARIABLES_OPENMETEO:
            valor = datos_openmeteo[variable][i]
            fila[variable] = float(valor) if valor is not None else None
        observaciones_openmeteo.append(fila)

if len(observaciones_openmeteo) < CANTIDAD_OBJETIVO:
    raise ValueError(
        f"OpenMeteo entregó {len(observaciones_openmeteo)} observaciones; "
        f"se requieren {CANTIDAD_OBJETIVO}."
    )

# Tomar 100 filas a intervalos regulares dentro de las últimas siete jornadas.
indices = [
    round(i * (len(observaciones_openmeteo) - 1) / (CANTIDAD_OBJETIVO - 1))
    for i in range(CANTIDAD_OBJETIVO)
]
registros_openmeteo = [observaciones_openmeteo[i] for i in indices]

esquema_openmeteo = StructType([
    StructField("tiempo", TimestampType(), False),
    StructField("temperature_2m", DoubleType(), True),
    StructField("relative_humidity_2m", DoubleType(), True),
    StructField("precipitation", DoubleType(), True),
    StructField("wind_speed_10m", DoubleType(), True),
    StructField("wind_direction_10m", DoubleType(), True),
    StructField("uv_index", DoubleType(), True)
])

df_openmeteo = spark.createDataFrame(
    registros_openmeteo, schema=esquema_openmeteo
)
print("Registros OpenMeteo:", df_openmeteo.count())
df_openmeteo.show(5, truncate=False)
df_openmeteo.printSchema()

## 4. Ingesta de datos — AQICN

Se mantiene el procedimiento del docente: 100 consultas con un segundo de separación. Se conserva la fecha de medición de la API y la hora de consulta; pueden existir 100 lecturas de una misma medición.

Al ejecutar, introduce un token AQICN para Santiago. También puede definirse en la variable de entorno `AQICN_TOKEN`. Enter conserva el token `demo` del ejemplo: permite probar el código, pero no acredita una medición de Santiago si devuelve otra estación. El nombre y las coordenadas recibidas se muestran antes de continuar.

Los valores de `iaqi` son índices individuales AQI publicados por **World Air Quality Index**; no se presentan como concentraciones en µg/m³. Los contaminantes no informados permanecen nulos.

In [ ]:
TOKEN_AQICN = (
    os.environ.get("AQICN_TOKEN")
    or getpass("Token AQICN para Santiago (Enter = demo del docente): ").strip()
    or "demo"
)
CIUDAD_AQICN = "santiago"
URL_AQICN = f"https://api.waqi.info/feed/{CIUDAD_AQICN}/"

registros_aqicn = []
region_aqicn_validada = True
for intento in range(CANTIDAD_OBJETIVO):
    respuesta = requests.get(URL_AQICN, params={"token": TOKEN_AQICN}, timeout=30)
    if respuesta.status_code != 200:
        raise RuntimeError(f"AQICN respondió HTTP {respuesta.status_code}. Revisa el acceso.")
    datos = respuesta.json()
    if datos.get("status") != "ok":
        raise ValueError("AQICN no autorizó la consulta. Revisa el token y la ciudad.")

    informacion = datos["data"]
    estacion = informacion.get("city", {})
    nombre_estacion = estacion.get("name", "Sin nombre")
    ciudad_coincide = CIUDAD_AQICN.casefold() in nombre_estacion.casefold()
    region_aqicn_validada = region_aqicn_validada and ciudad_coincide
    if intento == 0:
        print("Estación recibida:", nombre_estacion)
        print("Coordenadas:", estacion.get("geo"))
        print("Fuente:", estacion.get("url"))
        if not ciudad_coincide:
            print("PENDIENTE: la respuesta no identifica Santiago. Revisa la estación y el token.")
            print("Estas lecturas sirven para demostrar el flujo; no acreditan el requisito regional.")

    iaqi = informacion.get("iaqi", {})
    registros_aqicn.append({
        "estacion_aqicn": nombre_estacion,
        "tiempo_medicion": informacion.get("time", {}).get("iso")
            or informacion.get("time", {}).get("s"),
        "hora_consulta": datetime.now(ZoneInfo(ZONA_HORARIA)),
        "PM2_5": iaqi.get("pm25", {}).get("v"),
        "PM10": iaqi.get("pm10", {}).get("v"),
        "O3": iaqi.get("o3", {}).get("v"),
        "NO2": iaqi.get("no2", {}).get("v"),
        "SO2": iaqi.get("so2", {}).get("v")
    })
    if intento < CANTIDAD_OBJETIVO - 1:
        time.sleep(1)

esquema_aqicn = StructType([
    StructField("estacion_aqicn", StringType(), False),
    StructField("tiempo_medicion", StringType(), True),
    StructField("hora_consulta", TimestampType(), False),
    StructField("PM2_5", DoubleType(), True),
    StructField("PM10", DoubleType(), True),
    StructField("O3", DoubleType(), True),
    StructField("NO2", DoubleType(), True),
    StructField("SO2", DoubleType(), True)
])
for fila in registros_aqicn:
    for variable in ["PM2_5", "PM10", "O3", "NO2", "SO2"]:
        if fila[variable] is not None:
            fila[variable] = float(fila[variable])

df_aqicn = spark.createDataFrame(registros_aqicn, schema=esquema_aqicn)
print("Registros AQICN:", df_aqicn.count())
print("Mediciones distintas informadas:", df_aqicn.select("estacion_aqicn", "tiempo_medicion").distinct().count())
df_aqicn.show(5, truncate=False)
df_aqicn.printSchema()

## 5. Ingesta de datos — MongoDB (ESP32)

Se mantiene el host, la base y la colección del ejemplo docente. El usuario y la contraseña se ingresan al ejecutar, para no dejar credenciales escritas en el notebook.

El ejemplo entregado por el docente registró 28 documentos en su ejecución guardada. El notebook toma hasta 100 documentos disponibles y muestra el conteo real; si la colección aún tiene menos de 100, no inventa filas.

In [ ]:
usuario_mongo = os.environ.get("MONGO_USER") or input("Usuario de MongoDB entregado para el curso: ").strip()
clave_mongo = os.environ.get("MONGO_PASSWORD") or getpass("Contraseña de MongoDB: ")

from urllib.parse import quote_plus
mongo_uri = (
    f"mongodb+srv://{quote_plus(usuario_mongo)}:{quote_plus(clave_mongo)}"
    "@mgtt.yr90s39.mongodb.net/?appName=MGTT"
)

base_mongo = "espdb"
coleccion_mongo = "datos"

df_mongo_origen = (
    spark.read.format("com.mongodb.spark.sql.DefaultSource")
    .option("uri", mongo_uri)
    .option("database", base_mongo)
    .option("collection", coleccion_mongo)
    .load()
)

# Nombres observados en el esquema del ejemplo docente:
# altitudFt, presionAtm, ppmCO y fecha.
df_mongo_bpm = (
    df_mongo_origen
    .select(
        col("_id.oid").alias("id_mongo"),
        col("altitudFt").cast("double").alias("alturaft"),
        col("presionAtm").cast("double").alias("presion_atmosferica"),
        col("ppmCO").cast("double").alias("monoxido_carbono"),
        col("fecha").alias("fecha_mongo")
    )
    .orderBy(col("fecha_mongo").desc(), col("id_mongo").desc())
    .limit(CANTIDAD_OBJETIVO)
)

print("Registros MongoDB disponibles (máximo 100):", df_mongo_bpm.count())
df_mongo_bpm.show(5, truncate=False)
df_mongo_bpm.printSchema()

## 6. Ingesta de datos — MySQL mediante API REST

Se conserva la URL y el procedimiento del ejemplo docente: la base MySQL se consulta mediante el endpoint PHP que entrega JSON. Se toman los primeros 100 registros si están disponibles.

In [ ]:
URL_MYSQL = "http://mgtt.comunidadingenieria.cl/obtener_datos.php"

respuesta_mysql = requests.get(URL_MYSQL, timeout=30)
respuesta_mysql.raise_for_status()
datos_mysql = respuesta_mysql.json()

if not isinstance(datos_mysql, list):
    raise ValueError("La API de MySQL no devolvió una lista JSON.")

registros_mysql = datos_mysql[:CANTIDAD_OBJETIVO]

esquema_mysql = StructType([
    StructField("id_mysql", IntegerType(), True),
    StructField("temperatura_interior", DoubleType(), True),
    StructField("humedad", DoubleType(), True),
    StructField("fecha_mysql", StringType(), True)
])

filas_mysql = [
    (
        int(fila["id"]),
        float(fila["temperatura"]),
        float(fila["humedad"]),
        str(fila["fecha"])
    )
    for fila in registros_mysql
]

df_mysql_dht = spark.createDataFrame(filas_mysql, schema=esquema_mysql)
print("Registros MySQL disponibles (máximo 100):", df_mysql_dht.count())
df_mysql_dht.show(5, truncate=False)
df_mysql_dht.printSchema()

## 7. Control de cantidad de registros

La rúbrica pide 100 registros por fuente. Este control muestra qué fuentes alcanzan el requisito con los datos disponibles. Las filas ausentes no se generan artificialmente.

In [ ]:
conteos_fuentes = {
    "df_openmeteo": df_openmeteo.count(),
    "df_aqicn": df_aqicn.count(),
    "df_mongo_bpm": df_mongo_bpm.count(),
    "df_mysql_dht": df_mysql_dht.count()
}
for nombre, cantidad in conteos_fuentes.items():
    estado = "CUMPLE CANTIDAD" if cantidad == CANTIDAD_OBJETIVO else "PENDIENTE"
    print(f"{nombre}: {cantidad} registros — {estado}")

if any(cantidad != CANTIDAD_OBJETIVO for cantidad in conteos_fuentes.values()):
    print("Solicitar al docente acceso a los registros faltantes. No se generan filas de reemplazo.")
print("Región AQICN:", "Santiago identificada" if region_aqicn_validada else "PENDIENTE de validar")

## 8. Preprocesamiento y transformación

### 8.1 Conversión de altura de pies a metros

Se aplica el factor indicado en la pauta: **1 pie = 0,3048 metros**.

In [ ]:
df_mongo_bpm = df_mongo_bpm.withColumn(
    "altura_metros",
    spark_round(col("alturaft") * 0.3048, 2)
)

df_mongo_bpm.select(
    "alturaft", "altura_metros", "presion_atmosferica", "monoxido_carbono"
).show(10, truncate=False)

### 8.2 Clasificación de la condición climática

Se aplican los umbrales exactos de la rúbrica. Una temperatura ≤30 °C corresponde a `CLIMA MODERADO`, incluso si la humedad está ausente, porque esa regla solo depende de la temperatura. Cuando no se cumple ninguna condición, se asigna `CONDICION DESCONOCIDA`.

In [ ]:
df_openmeteo = df_openmeteo.withColumn(
    "condicion_clima",
    when(
        (col("temperature_2m") > 30) &
        (col("relative_humidity_2m") > 70),
        "CALOR Y HUMEDAD"
    ).when(
        (col("temperature_2m") > 30) &
        (col("relative_humidity_2m") <= 70),
        "CALOR SECO"
    ).when(
        col("temperature_2m") <= 30,
        "CLIMA MODERADO"
    ).otherwise("CONDICION DESCONOCIDA")
)

df_openmeteo.select(
    "temperature_2m", "relative_humidity_2m", "condicion_clima"
).show(10, truncate=False)

## 9. Integración de las cuatro fuentes

Las fuentes no comparten una clave natural. Se ordena cada una por fecha, se asigna un índice de fila y se unen los índices, como permite la pauta. Los identificadores de MongoDB y MySQL resuelven empates de fecha.

La unión es **externa completa (`full_outer`)**: si una fuente tiene menos filas, se conservan las demás y sus campos quedan nulos donde falta información. Se muestra el número esperado de filas y la cantidad real resultante.

**Supuesto y límite:** la fila 1 se empareja con la fila 1 de las otras fuentes. Es una integración posicional didáctica; no demuestra simultaneidad ni una relación causal entre clima y sensores. Se conservan las fechas y la estación de origen. Las consultas C1–C7 se realizan sobre cada fuente, sin usar estas asociaciones para sacar conclusiones ambientales.

In [ ]:
def agregar_indice_temporal(dataframe, columna_fecha, *desempates):
    columnas_orden = [col(columna_fecha).asc()] + [col(c).asc() for c in desempates]
    return dataframe.withColumn("indice", row_number().over(Window.orderBy(*columnas_orden)))

openmeteo_indice = agregar_indice_temporal(df_openmeteo, "tiempo")
aqicn_indice = agregar_indice_temporal(df_aqicn, "hora_consulta")
mongo_indice = agregar_indice_temporal(df_mongo_bpm, "fecha_mongo", "id_mongo")
mysql_indice = df_mysql_dht.withColumn("fecha_mysql_ts", to_timestamp(col("fecha_mysql")))
mysql_indice = agregar_indice_temporal(mysql_indice, "fecha_mysql_ts", "id_mysql")

df_unificado = (
    openmeteo_indice
    .join(aqicn_indice, on="indice", how="full_outer")
    .join(mongo_indice, on="indice", how="full_outer")
    .join(mysql_indice, on="indice", how="full_outer")
    .orderBy("indice")
)
filas_integradas = df_unificado.count()
filas_esperadas = max(conteos_fuentes.values())
print("Filas esperadas:", filas_esperadas, "| Filas integradas:", filas_integradas)
assert filas_integradas == filas_esperadas, "Revisar los índices de la integración."
df_unificado.show(10, truncate=False)
df_unificado.printSchema()

## 10. Tablas temporales para Spark SQL

Se crea una vista temporal por cada DataFrame fuente, como pide la instrucción.

In [ ]:
df_openmeteo.createOrReplaceTempView("df_openmeteo")
df_aqicn.createOrReplaceTempView("df_aqicn")
df_mongo_bpm.createOrReplaceTempView("df_mongo_bpm")
df_mysql_dht.createOrReplaceTempView("df_mysql_dht")

print("Vistas temporales creadas: df_openmeteo, df_aqicn, df_mongo_bpm, df_mysql_dht")

## 11. Consultas requeridas

### C1 — Diez temperaturas más altas de OpenMeteo

In [ ]:
consulta_c1 = spark.sql("""
    SELECT tiempo, temperature_2m
    FROM df_openmeteo
    ORDER BY temperature_2m DESC NULLS LAST, tiempo ASC
    LIMIT 10
""")
consulta_c1.show(truncate=False)

### C2 — Humedad relativa promedio para un día y mes

Se elige la fecha de la última observación de la muestra. Se conserva también el año para evitar mezclar jornadas de años distintos. El promedio se refiere a las observaciones seleccionadas de ese día.

In [ ]:
fecha_objetivo = registros_openmeteo[-1]["tiempo"].date().isoformat()
print("Fecha consultada:", fecha_objetivo)
consulta_c2 = spark.sql(f"""
    SELECT
        to_date(tiempo) AS fecha,
        COUNT(relative_humidity_2m) AS observaciones_validas,
        ROUND(AVG(relative_humidity_2m), 2) AS humedad_promedio
    FROM df_openmeteo
    WHERE to_date(tiempo) = DATE '{fecha_objetivo}'
    GROUP BY to_date(tiempo)
""")
consulta_c2.show(truncate=False)

### C3 — Registros con precipitación superior a 2 mm durante la última semana

Se utiliza el mismo intervalo de siete días fijado al descargar OpenMeteo. Así, volver a ejecutar la consulta no desplaza el período respecto de la captura. El resultado cuenta registros de la **muestra de 100 observaciones**, no todas las horas de la ciudad ni el número de eventos de lluvia.

In [ ]:
print("Intervalo consultado:", inicio_semana.isoformat(), "a", ahora_santiago.isoformat())
consulta_c3 = spark.sql(f"""
    SELECT COUNT(*) AS registros_sobre_2_mm
    FROM df_openmeteo
    WHERE tiempo BETWEEN TIMESTAMP '{inicio_semana.isoformat()}'
                     AND TIMESTAMP '{ahora_santiago.isoformat()}'
      AND precipitation > 2
""")
consulta_c3.show(truncate=False)

### C4 — Promedios de PM2.5, PM10, O3, NO2 y SO2

`AVG` considera los valores disponibles. Un contaminante totalmente ausente produce NULL; no se reemplaza por cero. Si AQICN repite una medición, estos promedios describen las lecturas capturadas, sin constituir una serie histórica de calidad del aire.

In [ ]:
consulta_c4 = spark.sql("""
    SELECT
        ROUND(AVG(PM2_5), 2) AS promedio_PM2_5,
        ROUND(AVG(PM10), 2) AS promedio_PM10,
        ROUND(AVG(O3), 2) AS promedio_O3,
        ROUND(AVG(NO2), 2) AS promedio_NO2,
        ROUND(AVG(SO2), 2) AS promedio_SO2
    FROM df_aqicn
""")
consulta_c4.show(truncate=False)

### C5 — Cantidad de registros con NO2 superior a 4

In [ ]:
consulta_c5 = spark.sql("""
    SELECT COUNT(*) AS registros_NO2_mayor_4
    FROM df_aqicn
    WHERE NO2 > 4
""")
consulta_c5.show(truncate=False)

### C6 — Diez registros de MongoDB, incluyendo altura en metros

In [ ]:
consulta_c6 = spark.sql("""
    SELECT
        alturaft,
        altura_metros,
        presion_atmosferica,
        monoxido_carbono
    FROM df_mongo_bpm
    ORDER BY fecha_mongo ASC, id_mongo ASC
    LIMIT 10
""")
consulta_c6.show(truncate=False)

### C7 — Diez registros de OpenMeteo, incluyendo condición climática

In [ ]:
consulta_c7 = spark.sql("""
    SELECT
        tiempo,
        temperature_2m,
        relative_humidity_2m,
        condicion_clima
    FROM df_openmeteo
    ORDER BY tiempo ASC
    LIMIT 10
""")
consulta_c7.show(truncate=False)

## 12. Resumen de ejecución

Este resumen permite revisar rápidamente que existan los cuatro DataFrames, la integración y las siete consultas solicitadas.

In [ ]:
print("RESUMEN DE LA SUMATIVA 3")
for nombre, cantidad in conteos_fuentes.items():
    print(nombre, ":", cantidad, "filas")
print("DataFrame integrado:", df_unificado.count(), "filas")
print("Estación AQICN:", nombre_estacion)
print("Región AQICN:", "Santiago identificada" if region_aqicn_validada else "PENDIENTE")
print("Consultas ejecutadas: C1, C2, C3, C4, C5, C6 y C7")
print("C3 — Lecturas con más de 2 mm:", consulta_c3.first()["registros_sobre_2_mm"])
print("C5 — Lecturas con NO2 mayor a 4:", consulta_c5.first()["registros_NO2_mayor_4"])

## Lista de entrega

- [ ] Ejecutar el notebook desde el inicio con **Entorno de ejecución → Ejecutar todo**.
- [ ] Revisar que cada fuente tenga 100 registros y guardar las salidas visibles.
- [ ] Confirmar que AQICN corresponda a Santiago; el token demo por sí solo no acredita este requisito.
- [ ] Confirmar que las columnas de los DataFrames y las siete consultas aparecen sin errores.
- [ ] Grabar el video de 7 a 10 minutos con participación de todos los integrantes.
- [ ] Incluir en el video las conexiones, las transformaciones, la decisión de integración y los resultados.
- [ ] Nombrar el notebook `mcdi502_s3_grupo7.ipynb` y el video `mcdi502_s3_grupo7_video.mp4`.
- [ ] Comprimir ambos archivos en un único ZIP para la entrega.

**Estado de esta versión:** el flujo completo requiere ejecutar Colab con el acceso real a MongoDB y el token regional de AQICN. No se incluyen salidas simuladas de esas conexiones.

**Antes de entregar:** si MongoDB o MySQL siguen entregando menos de 100 registros, el código lo dejará visible. Se requiere que el origen disponga de 100 registros; el notebook no crea filas ficticias.

## Referencias

- Rúbrica Sumativa 3 — Taller II, MCDI502, UNAB.
- Ejemplos docentes: BaseEjemploOpenMeteo, BaseEjemploAQICN, BaseEjemploMongoDB y BaseEjemploMysqlPHP_APIRest.
- Apache Spark 3.5.7: https://spark.apache.org/docs/3.5.7/
- Open-Meteo: https://open-meteo.com/en/docs
- World Air Quality Index y entidad que informa la estación: https://aqicn.org/api/
